In [ ]:
import os

import snowflake.connector
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T

In [ ]:
connection_params = {
    "account": os.environ.get("SNOWFLAKE_ACCOUNT"),
    "password": os.environ.get("SNOWFLAKE_PASSWORD"),
    "database": os.environ.get("SNOWFLAKE_DATABASE"),
    "schema": os.environ.get("SNOWFLAKE_SCHEMA"),
    "warehouse": os.environ.get("SNOWFLAKE_WAREHOUSE"),
    "user": os.environ.get("SNOWFLAKE_USER"),
    "role": os.environ.get("SNOWFLAKE_ROLE"),
}

In [ ]:
con = snowflake.connector.connect(**connection_params)
session = Session.builder.config("connection", con).getOrCreate()

In [ ]:
session.sql("USE WAREHOUSE SNOWADHOC;").show()

In [ ]:
# case_chronicles = session.table("CX360.CASE_CHRONICLES")
case_chronicles = session.table("SUPPORT.SHARE_TO_AGENTS.CASE_CHRONICLES")
pcs = F.col("PRIME_CASE_STRUCTURED")
content = pcs["content"]["initial_post"]["content"]
metadata = pcs["metadata"]["case"]


transformed_cases = case_chronicles.select(
    "CASE_ID",
    "CASE_NUMBER",
    content["subject"].alias("content_subject"),
    content["description"].alias("content_description"),
    pcs["content"]["initial_post"]["timestamp"].cast(T.TimestampType()).alias("initial_post_created_at"),
    metadata["account_name"].cast(T.StringType()).alias("account_name"),
    metadata["case_number"].cast(T.StringType()).alias("case_number"),
    metadata["closed_at"].cast(T.TimestampType()).alias("closed_at"),
    metadata["current_severity"].cast(T.StringType()).alias("current_severity"),
    metadata["is_ascii"].cast(T.BooleanType()).alias("is_ascii"),
    metadata["is_priority_support_entitlement"].cast(T.BooleanType()).alias("is_priority_support_entitlement"),
    metadata["peak_severity"].cast(T.StringType()).alias("peak_severity"),
    metadata["sfdc_account_id"].cast(T.StringType()).alias("sfdc_account_id"),
    metadata["sfdc_case_id"].cast(T.StringType()).alias("sfdc_case_id"),
    metadata["total_comments"].cast(T.LongType()).alias("total_comments"),
)

transformed_cases.write.save_as_table("AI_FDE.CX360.CASES", mode="overwrite")